# Conatus Eidos — Fase 0: baseline do Qwen2.5-Coder-7B no harness

Roda os 100 casos de `eidos/eval_cases.jsonl` com o coder CRU (sem adapter).
GPU recomendada: **L4** (24GB). Rodada completa: ~1,5-3h. Use a célula de
rodada parcial primeiro pra ter sinal em minutos.


In [ ]:
# 1) Repo na branch conatus-eidos + login HF
import os
from google.colab import userdata
from huggingface_hub import login

GH_TOKEN = userdata.get("GH_TOKEN")
GH_USER = "devlucascfarias"
REPO_NAME = "Conatus-Phronesis"
REPO = f"/content/{REPO_NAME}"
REPO_URL = f"https://{GH_TOKEN}@github.com/{GH_USER}/{REPO_NAME}.git"

if os.path.isdir(f"{REPO}/.git"):
    !cd {REPO} && git fetch && git checkout conatus-eidos && git pull --ff-only
else:
    !git clone -b conatus-eidos {REPO_URL} {REPO}

del GH_TOKEN, REPO_URL
login(token=userdata.get("HF_TOKEN"))


In [ ]:
# 2) Node >= 18 (o Colab costuma ter; se nao tiver, instala) + deps do template
import subprocess
ok = False
try:
    v = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    ok = int(v.lstrip("v").split(".")[0]) >= 18
    print("node:", v)
except FileNotFoundError:
    pass
if not ok:
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get install -y nodejs > /dev/null
    !node --version

%cd {REPO}/eidos/template_app
!npm install --no-audit --no-fund 2>&1 | tail -2
%cd {REPO}


In [ ]:
# 3) QA rapido dos casos (tsc real em 46 setups, ~4-6 min) — opcional mas recomendado 1x
!cd {REPO} && python eidos/verify_cases.py --skip-build


In [ ]:
# 4) Ambiente Python do modelo
%pip install -q -U transformers accelerate


In [ ]:
# 5) SINAL RAPIDO: so a familia fix-build (30 casos) — ~30-60 min
!cd {REPO} && python eidos/run_eval.py --model Qwen/Qwen2.5-Coder-7B-Instruct --family fix-build


In [ ]:
# 6) RODADA COMPLETA: os 100 casos (sobrescreve o metrics.json da parcial)
!cd {REPO} && python eidos/run_eval.py --model Qwen/Qwen2.5-Coder-7B-Instruct


In [ ]:
# 7) Resultado + empacotar transcripts pra analise
import json, shutil
print(json.dumps(json.load(open(f"{REPO}/eidos/results/metrics.json")), indent=2, ensure_ascii=False))
shutil.make_archive("/content/eidos_baseline_transcripts", "zip", f"{REPO}/eidos/results")
print("
zip pronto: /content/eidos_baseline_transcripts.zip (baixa e me manda a analise)")


In [ ]:
# 8) Versionar o metrics.json do baseline no repo (transcripts ficam so no zip)
!cd {REPO} && mkdir -p eidos/results && cp eidos/results/metrics.json eidos/results/baseline_7b_metrics.json   && git add eidos/results/baseline_7b_metrics.json   && git -c user.name="colab" -c user.email="colab@local" commit -m "Eidos Fase 0: metricas do baseline Qwen2.5-Coder-7B"   && git push origin conatus-eidos
